# 神经网络三大核心任务学习
>目标：彻底搞懂 `backward()`、`update_params()`、`shape_check()` 这三个函数——在做什么、为什么这样做、怎么一行行写出来。
>前置：你已经写完了 `forward()`（前向传播）和损失函数（比如交叉熵）。

## 0. 把场景固定下来（以后都对着这张图看公式）

本笔记以最经典的「两层全连接网络 + 多分类」为例，这也是绝大多数 `nn.py` 的标准形态：

```
X ─► Linear(W1,b1) ─► Z1 ─► ReLU ─► A1 ─► Linear(W2,b2) ─► Z2 ─► Softmax ─► Ŷ ─► CrossEntropy(Ŷ,Y) ─► L
```

记号约定（贯穿整篇笔记）：

| 记号     | 含义                | 形状         |
| -------- | ------------------- | ------------ |
| `X`      | 一个 batch 的输入   | `(m, n_x)`   |
| `Y`      | 真实标签（one-hot） | `(m, n_y)`   |
| `W1`     | 第一层权重          | `(n_x, n_h)` |
| `b1`     | 第一层偏置          | `(1, n_h)`   |
| `Z1`     | 第一层线性输出      | `(m, n_h)`   |
| `A1`     | 第一层激活输出      | `(m, n_h)`   |
| `W2`     | 第二层权重          | `(n_h, n_y)` |
| `b2`     | 第二层偏置          | `(1, n_y)`   |
| `Z2`     | 第二层线性输出      | `(m, n_y)`   |
| `Ŷ` (A2) | Softmax 预测概率    | `(m, n_y)`   |
| `L`      | 损失（标量）        | `()`         |

> `m` 是 batch size，`n_x / n_h / n_y` 分别是输入/隐藏/输出维度。

## 1. 前置：链式法则到底在说啥

**反向传播 = 链式法则 + 按层从后往前走。**

一句话：要算 `∂L/∂W1`，就把它拆成一串"前面已经算过的梯度"连乘起来。

$$\frac{\partial L}{\partial W_1} = \frac{\partial L}{\partial Z_2} \cdot \frac{\partial Z_2}{\partial A_1} \cdot \frac{\partial A_1}{\partial Z_1} \cdot \frac{\partial Z_1}{\partial W_1}$$

所以只要按**从后往前**一步步推，每一步的结果都能被下一步复用——这就是 `backward()` 的核心。

## 2. 任务一：实现完整的 `backward()`

### 2.1 每一层的梯度公式（推一次，以后照抄）

**Step ①：Softmax + CrossEntropy 合并求导**（非常漂亮的结论，必须记住）

$$\frac{\partial L}{\partial Z_2} = \frac{1}{m}(\hat Y - Y)$$

代码里记作 `dZ2`，形状 `(m, n_y)`。
为什么这么简洁？因为 softmax 的雅可比矩阵与交叉熵的梯度向量相乘后，大量项两两抵消，最后就只剩 `Ŷ − Y`。现在记结论即可，推导可以以后深究。

**Step ②：第二层参数的梯度**

$$\frac{\partial L}{\partial W_2} = A_1^{\mathsf T} \cdot dZ_2,\qquad \frac{\partial L}{\partial b_2} = \sum_{\text{batch}} dZ_2$$

**Step ③：把误差传回隐藏层**

$$dA_1 = dZ_2 \cdot W_2^{\mathsf T},\qquad dZ_1 = dA_1 \odot g'(Z_1)$$

其中 `g'(Z1)` 是激活函数的导数。对 ReLU：`g'(z) = 1 if z > 0 else 0`。
`⊙` 是 **element-wise 乘法**（Hadamard 积），**不是矩阵乘法**——这点极易写错。

**Step ④：第一层参数的梯度**

$$\frac{\partial L}{\partial W_1} = X^{\mathsf T} \cdot dZ_1,\qquad \frac{\partial L}{\partial b_1} = \sum_{\text{batch}} dZ_1$$

### 2.2 代码实现（每行注释对应哪条公式）

In [ ]:
import numpy as np



def backward(self, X, Y):
    m = X.shape[0]                            # batch size

    # ---------- 输出层 ----------
    dZ2 = (self.A2 - Y) / m                   # ∂L/∂Z2 = (Ŷ − Y) / m
    dW2 = self.A1.T @ dZ2                     # ∂L/∂W2 = A1ᵀ · dZ2
    db2 = np.sum(dZ2, axis=0, keepdims=True)  # ∂L/∂b2 = Σ dZ2（沿 batch 维求和）

    # ---------- 隐藏层 ----------
    dA1 = dZ2 @ self.W2.T                     # ∂L/∂A1 = dZ2 · W2ᵀ
    dZ1 = dA1 * relu_deriv(self.Z1)           # ∂L/∂Z1 = dA1 ⊙ g'(Z1)   element-wise！
    dW1 = X.T @ dZ1                           # ∂L/∂W1 = Xᵀ · dZ1
    db1 = np.sum(dZ1, axis=0, keepdims=True)  # ∂L/∂b1 = Σ dZ1

    # 梯度统一存起来，交给 update_params 用
    self.grads = {"dW1": dW1, "db1": db1,
                  "dW2": dW2, "db2": db2}
    return self.grads


def relu_deriv(Z):
    # ReLU 导数：z > 0 → 1，否则 → 0
    return (Z > 0).astype(Z.dtype)

### 2.3 三个最常见的坑

1. **`/ m` 只除一次**：要么放 `dZ2` 里（推荐，一次到位），要么每层的 `dW/db` 里单独除，**不要两边都除**。
2. **`*` vs `@`**：`dZ1 = dA1 * g'(Z1)` 是逐元素乘；`dW2 = A1.T @ dZ2` 是矩阵乘。搞混了 shape 会直接炸。
3. **`keepdims=True`**：`db` 求和后如果掉一维变成 `(n,)`，后续 `b -= lr*db` 会被广播搞歪。保持 `(1, n)` 这种二维形状最安全。

## 3. 任务二：实现 `update_params()`（SGD）

### 3.1 原理：梯度下降在做啥

参数 `θ`（泛指 `W1/b1/W2/b2`）沿着梯度的**反方向**迈一小步，损失就会下降。

$$\theta \leftarrow \theta - \alpha \cdot \frac{\partial L}{\partial \theta}$$

其中 `α` 是学习率（learning rate, `lr`）。

直觉解释：

- 梯度指向"损失增大最快"的方向；我们反着走，就是"下山"。
- `α` 太大→步子迈过头，loss 震荡甚至爆炸；太小→训练慢得离谱。

### 3.2 代码实现

In [ ]:
def update_params(self,lr):
     # 对每个参数执行：θ ← θ − lr · dθ
    self.W1-=lr*self.grads["dW1"]
    self.b1-=lr*self.grads["db1"]
    self.W2-=lr*self.grads["dW2"]
    self.b2-=lr*self.grads["db2"]

就这。朴素版 SGD 就这四行。

### 3.3 小细节

- 用 `-=` 原地减，避免额外分配内存。
- `lr` 一般在 `1e-3 ~ 1e-1`；MNIST 这类简单任务 `0.01~0.1` 都行。
- 以后想升级 Momentum / Adam，只需在这个函数里加「速度缓存」或「一/二阶矩估计」，**接口签名不用变**——这就是解耦的好处。

## 4. 任务三：实现 `shape_check()`

### 4.1 为什么需要它

反向传播最容易出的 bug **不是算错公式，而是 shape 对不上**。
NumPy 的广播机制会"贴心地"把本该报错的地方悄悄算出一个错误结果，模型照跑，loss 就是不降——这是初学者最痛苦的 bug 源头。

`shape_check()` 的作用：**每次 backward 之后，立刻断言每个梯度的 shape 与它对应参数的 shape 完全一致**。一旦失配，程序立刻炸，错误位置一目了然。

### 4.2 核心规则（一句话：**梯度 shape 必须等于参数 shape**）

| 参数 | 形状         | 梯度  | 形状必须     |
| ---- | ------------ | ----- | ------------ |
| `W1` | `(n_x, n_h)` | `dW1` | `(n_x, n_h)` |
| `b1` | `(1, n_h)`   | `db1` | `(1, n_h)`   |
| `W2` | `(n_h, n_y)` | `dW2` | `(n_h, n_y)` |
| `b2` | `(1, n_y)`   | `db2` | `(1, n_y)`   |

为什么必须一致？因为 `update_params` 要做 `W -= lr * dW`：两边形状不一致，要么广播出错，要么直接报错。

### 4.3 代码实现

In [ ]:
def shape_check(self):
    pairs = [
        ("W1", self.W1, "dW1", self.grads["dW1"]),
        ("b1", self.b1, "db1", self.grads["db1"]),
        ("W2", self.W2, "dW2", self.grads["dW2"]),
        ("b2", self.b2, "db2", self.grads["db2"]),
    ]
    for pname, p, gname, g in pairs:
        assert p.shape == g.shape, (
            f"[shape_check] {gname}{g.shape} != {pname}{p.shape}"
        )

`assert cond, msg` 这种写法：失败时会把 `msg` 原样打出来，方便你瞬间定位是哪一对 shape 没对上。

### 4.4 进阶：还可以多检查什么

- **中间量 shape**：`assert Z1.shape == (m, n_h)`、`assert A1.shape == Z1.shape`。
- **批次对齐**：`assert X.shape[0] == Y.shape[0] == m`。
- **数值安全**：`assert not np.isnan(loss)`（训练跑飞时第一时间发现）。
- **debug 开关**：用 `if self.debug: self.shape_check()` 包一层，正式训练时关掉可省时间。

---


## 5. 三个函数怎么串起来：一个完整 epoch 长这样

In [ ]:
for epoch in range(num_epochs):
    for X_batch, Y_batch in loader:
        # 1. 前向：算预测和损失
        Y_hat = model.forward(X_batch)
        loss  = cross_entropy(Y_hat, Y_batch)

        # 2. 反向：算梯度
        model.backward(X_batch, Y_batch)

        # 3. 调试：shape 对不对
        model.shape_check()

        # 4. 更新参数
        model.update_params(lr=0.01)

你会发现一条很清晰的脉络：
**forward 存中间量 → backward 用这些中间量算梯度 → update 用梯度改参数**。
这三步职责完全解耦——这正是 PyTorch / TensorFlow 等框架的 API 设计哲学（`loss.backward()` + `optimizer.step()`）。你手写一遍，以后用框架就不再是黑箱。